# 其他实验代码（非正式提交）

**正式提交请用 `Mask_Student.ipynb`。本文件不要 Run All 去覆盖 `submission.csv`。**

这里收纳竞赛过程中还用过、但没进最终配方的代码，分块注释。各节独立，跑完最多写自己的 csv。

已确认无提升、因而不再保留可运行代码的尝试：V13 diversity 权重、naji blend、10 折 OOF 混进 5 折 meta、在同一份 OOF 上搜 C / 融合权重、rank 融合 / 分区 LR（相对 R3 只有 +1e-5 量级）。


## A. 自训 5 折 GBDT（原 `train.py`）

两套 LightGBM + 一套 XGBoost，`StratifiedKFold(5, shuffle=True, random_state=42)`，OOF 上网格搜非负凸组合。
AUC 只看排序，**不用** `scale_pos_weight`。写出 `submission_own_5fold.csv`，不改 `submission.csv`。


In [ ]:
from __future__ import annotations
import warnings
from itertools import product
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
ROOT = Path(".").resolve()
DATA_DIR = ROOT / "data"
N_FOLDS, SEED = 5, 42
TARGET = "addicted_label"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]
NUM_COLS = [
    "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "notifications_per_day",
    "app_opens_per_day", "weekend_screen_time",
]

def add_features_v1(df: pd.DataFrame) -> pd.DataFrame:
    """缺失指示 + 屏幕时间比率/交互。树模型保留 NaN，不做均值填充。"""
    out = df.copy()
    base = [c for c in NUM_COLS + CAT_COLS if c in out.columns]
    for c in base:
        out[f"{c}_isna"] = out[c].isna().astype(np.int8)
    out["n_missing"] = out[base].isna().sum(axis=1).astype(np.int8)
    d, s, g, w = out["daily_screen_time_hours"], out["social_media_hours"], out["gaming_hours"], out["work_study_hours"]
    sl, we, n, a = out["sleep_hours"], out["weekend_screen_time"], out["notifications_per_day"], out["app_opens_per_day"]
    leisure = s + g
    out["leisure_hours"] = leisure
    out["other_hours"] = d - leisure - w
    out["leisure_ratio"] = leisure / d
    out["social_ratio"] = s / d
    out["gaming_ratio"] = g / d
    out["work_ratio"] = w / d
    out["weekend_ratio"] = we / d
    out["weekend_delta"] = we - d
    out["sleep_deficit"] = 8.0 - sl
    out["screen_sleep_ratio"] = d / sl
    out["weekend_sleep_ratio"] = we / sl
    out["opens_per_hour"] = a / d
    out["notif_per_hour"] = n / d
    out["notif_per_open"] = n / a
    out["engagement"] = n * a
    out["log_notifications"] = np.log1p(n)
    out["log_app_opens"] = np.log1p(a)
    out["social_x_daily"] = s * d
    out["weekend_x_daily"] = we * d
    out["social_x_weekend"] = s * we
    out["leisure_x_weekend"] = leisure * we
    out["screen_sq"] = d * d
    out["weekend_sq"] = we * we
    out["stress_ord"] = out["stress_level"].map({"Low": 0.0, "Medium": 1.0, "High": 2.0})
    out["impact_bin"] = out["academic_work_impact"].map({"No": 0.0, "Yes": 1.0})
    for c in CAT_COLS:
        out[c] = out[c].astype("category")
    return out

LGB_A = dict(objective="binary", metric="auc", boosting_type="gbdt", learning_rate=0.03,
    num_leaves=64, max_depth=-1, min_child_samples=80, subsample=0.80, subsample_freq=1,
    colsample_bytree=0.55, reg_alpha=0.2, reg_lambda=1.5, max_bin=255, verbosity=-1,
    n_jobs=-1, seed=SEED, feature_fraction_seed=SEED, bagging_seed=SEED)
LGB_B = {**LGB_A, "num_leaves": 96, "learning_rate": 0.02, "colsample_bytree": 0.45,
    "min_child_samples": 120, "reg_lambda": 2.5, "seed": SEED + 17,
    "feature_fraction_seed": SEED + 17, "bagging_seed": SEED + 17}
XGB_P = dict(objective="binary:logistic", eval_metric="auc", learning_rate=0.03, max_depth=6,
    min_child_weight=20, subsample=0.80, colsample_bytree=0.55, reg_lambda=1.5, reg_alpha=0.2,
    max_bin=256, tree_method="hist", nthread=-1, seed=SEED)

def train_lgb(X_tr, y_tr, X_va, y_va, params, cat_cols):
    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols, free_raw_data=False)
    dvalid = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols, reference=dtrain, free_raw_data=False)
    model = lgb.train(params, dtrain, num_boost_round=4500, valid_sets=[dvalid], valid_names=["valid"],
        callbacks=[lgb.early_stopping(stopping_rounds=120, verbose=False), lgb.log_evaluation(period=400)])
    return model, model.predict(X_va, num_iteration=model.best_iteration)

def train_xgb(X_tr, y_tr, X_va, y_va, params):
    dtrain = xgb.DMatrix(X_tr, label=y_tr, enable_categorical=True)
    dvalid = xgb.DMatrix(X_va, label=y_va, enable_categorical=True)
    model = xgb.train(params, dtrain, num_boost_round=4500, evals=[(dvalid, "valid")],
        early_stopping_rounds=120, verbose_eval=400)
    return model, model.predict(dvalid, iteration_range=(0, model.best_iteration + 1))


### A2. 跑 5 折并融合

需要较长时间。OOF 大约 0.966 量级，明显低于公开库 stacking，所以没有进最终提交。


In [ ]:
# 若只想看代码、不训练，把下一行改成 False
RUN_OWN_5FOLD = False

if RUN_OWN_5FOLD:
    train = pd.read_csv(DATA_DIR / "train.csv")
    test = pd.read_csv(DATA_DIR / "test.csv")
    y = train[TARGET].astype(np.int32).values
    train_fe, test_fe = add_features_v1(train), add_features_v1(test)
    feats = [c for c in train_fe.columns if c not in {"id", TARGET}]
    cat_in = [c for c in CAT_COLS if c in feats]
    X, X_test = train_fe[feats], test_fe[feats]
    spec = {"lgb_a": ("lgb", LGB_A), "lgb_b": ("lgb", LGB_B), "xgb": ("xgb", XGB_P)}
    oof = {n: np.zeros(len(X)) for n in spec}
    te = {n: np.zeros(len(X_test)) for n in spec}
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        print(f"----- fold {fold} -----")
        for name, (kind, params) in spec.items():
            if kind == "lgb":
                model, pv = train_lgb(X.iloc[tr], y[tr], X.iloc[va], y[va], params, cat_in)
                pt = model.predict(X_test, num_iteration=model.best_iteration)
            else:
                model, pv = train_xgb(X.iloc[tr], y[tr], X.iloc[va], y[va], params)
                pt = model.predict(xgb.DMatrix(X_test, enable_categorical=True),
                                   iteration_range=(0, model.best_iteration + 1))
            oof[name][va] = pv
            te[name] += pt / N_FOLDS
            print(name, roc_auc_score(y[va], pv))
    mats = np.column_stack([oof[n] for n in spec])
    best_auc, best_w = -1.0, None
    grid = np.linspace(0, 1, 11)
    names = list(spec)
    for w0 in grid:
        for w1 in grid:
            rest = 1.0 - w0 - w1
            if rest < -1e-9:
                continue
            w = np.array([w0, w1, max(rest, 0.0)]); w = w / w.sum()
            a = roc_auc_score(y, mats @ w)
            if a > best_auc:
                best_auc, best_w = a, w
    print("blend", dict(zip(names, best_w)), best_auc)
    pred = np.clip(sum(best_w[i] * te[n] for i, n in enumerate(names)), 1e-7, 1 - 1e-7)
    pd.DataFrame({"id": test["id"], TARGET: pred}).to_csv(ROOT / "submission_own_5fold.csv", index=False)
    print("wrote submission_own_5fold.csv")
else:
    print("skip own 5-fold (set RUN_OWN_5FOLD = True to train)")


## B. 自训 10 折 GBDT（原 `train_v3.py` / `Model_SelfTrain_GBDT_10Fold.ipynb`）

预算/周期性特征 + 两种子 LightGBM + XGBoost + CatBoost，**10 折**。OOF 约 **0.96726**。

**不要**把 10 折 OOF 送进上面的 5 折元模型（折不一致）。写出 `submission_own_10fold.csv`。


In [ ]:
HOUR_COLS = [
    "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "weekend_screen_time",
]

def add_features_v3(df: pd.DataFrame) -> pd.DataFrame:
    """预算分解 + 小时列的 sin/cos/小数部分。3 列 LOO TE 在 holdout 上掉分，默认关掉。"""
    out = df.copy()
    d, s, g, w = out["daily_screen_time_hours"], out["social_media_hours"], out["gaming_hours"], out["work_study_hours"]
    we, n, a = out["weekend_screen_time"], out["notifications_per_day"], out["app_opens_per_day"]
    used = s + g + w
    slack = d - used
    out["budget_used"] = used
    out["budget_slack"] = slack
    out["budget_slack_ratio"] = slack / d
    out["budget_tight"] = (slack.abs() < 0.05).astype(np.float32)
    out["weekend_slack"] = we - (s + g)
    out["leisure_hours"] = s + g
    out["daily_x_social"] = d * s
    out["daily_x_weekend"] = d * we
    out["social_x_weekend"] = s * we
    for c in HOUR_COLS:
        x = out[c]
        out[f"{c}_sin"] = np.sin(2 * np.pi * x)
        out[f"{c}_cos"] = np.cos(2 * np.pi * x)
        out[f"{c}_frac"] = x - np.floor(x)
        out[f"{c}_d1"] = np.floor(x * 10.0) % 10.0
        out[f"{c}_log1p"] = np.log1p(x)
    out["notif_log1p"] = np.log1p(n)
    out["opens_log1p"] = np.log1p(a)
    out["notif_mod10"] = n % 10
    out["opens_mod10"] = a % 10
    out["stress_ord"] = out["stress_level"].map({"Low": 0.0, "Medium": 1.0, "High": 2.0})
    out["impact_bin"] = out["academic_work_impact"].map({"No": 0.0, "Yes": 1.0})
    gnd = out["gender"].astype("string").fillna("NA")
    st = out["stress_level"].astype("string").fillna("NA")
    out["gender_stress"] = (gnd + "_" + st).astype("category")
    for c in CAT_COLS:
        out[c] = out[c].astype("category")
    return out

def lgb_params_v3(seed: int) -> dict:
    return dict(objective="binary", metric="auc", boosting_type="gbdt", learning_rate=0.015,
        num_leaves=63, min_child_samples=40, subsample=0.80, subsample_freq=1,
        colsample_bytree=0.50, reg_alpha=0.1, reg_lambda=2.0, max_bin=1024, verbosity=-1,
        n_jobs=-1, seed=seed, feature_fraction_seed=seed, bagging_seed=seed)

XGB_V3 = dict(objective="binary:logistic", eval_metric="auc", learning_rate=0.015, max_depth=7,
    min_child_weight=10, subsample=0.80, colsample_bytree=0.50, reg_lambda=2.0, reg_alpha=0.1,
    max_bin=512, tree_method="hist", nthread=-1, seed=SEED)
CB_V3 = dict(loss_function="Logloss", eval_metric="AUC", iterations=12000, learning_rate=0.03,
    depth=6, l2_leaf_reg=4.0, bootstrap_type="Bernoulli", subsample=0.80, random_seed=SEED,
    early_stopping_rounds=200, verbose=1000, thread_count=-1, allow_writing_files=False)

RUN_OWN_10FOLD = False  # 数小时；需要 catboost

if RUN_OWN_10FOLD:
    try:
        from catboost import CatBoostClassifier, Pool
        HAS_CB = True
    except Exception:
        HAS_CB = False
        print("no catboost, skip cb")
    train = pd.read_csv(DATA_DIR / "train.csv")
    test = pd.read_csv(DATA_DIR / "test.csv")
    y = train[TARGET].astype(np.int32).values
    tr_fe, te_fe = add_features_v3(train), add_features_v3(test)
    feats = [c for c in tr_fe.columns if c not in {"id", TARGET}]
    cats = [c for c in CAT_COLS + ["gender_stress"] if c in feats]
    X, X_test = tr_fe[feats], te_fe[feats]
    spec = {f"lgb_{sd}": ("lgb", lgb_params_v3(sd)) for sd in (42, 2026)}
    spec["xgb"] = ("xgb", XGB_V3)
    if HAS_CB:
        spec["cb"] = ("cb", None)
    oof = {n: np.zeros(len(X)) for n in spec}
    te = {n: np.zeros(len(X_test)) for n in spec}
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        print("----- 10-fold", fold, "-----")
        for name, (kind, params) in spec.items():
            if kind == "lgb":
                model, pv = train_lgb(X.iloc[tr], y[tr], X.iloc[va], y[va], params, cats)
                # reuse 5-fold trainers but with v3 params / more rounds: override below if needed
                pt = model.predict(X_test, num_iteration=model.best_iteration)
            elif kind == "xgb":
                model, pv = train_xgb(X.iloc[tr], y[tr], X.iloc[va], y[va], params)
                pt = model.predict(xgb.DMatrix(X_test, enable_categorical=True),
                                   iteration_range=(0, model.best_iteration + 1))
            else:
                def _cb(frame):
                    o = frame.copy()
                    for c in cats:
                        o[c] = o[c].astype("string").fillna("NA")
                    return o
                model = CatBoostClassifier(**CB_V3)
                model.fit(Pool(_cb(X.iloc[tr]), y[tr], cat_features=cats),
                          eval_set=Pool(_cb(X.iloc[va]), y[va], cat_features=cats), use_best_model=True)
                pv = model.predict_proba(Pool(_cb(X.iloc[va]), cat_features=cats))[:, 1]
                pt = model.predict_proba(Pool(_cb(X_test), cat_features=cats))[:, 1]
            oof[name][va] = pv
            te[name] += pt / 10
            print(name, roc_auc_score(y[va], pv))
    names = list(spec)
    mats = np.column_stack([oof[n] for n in names])
    best_auc, best_w = -1.0, None
    for ws in product(np.linspace(0, 1, 11), repeat=len(names)):
        w = np.asarray(ws, dtype=np.float64)
        if w.sum() <= 0:
            continue
        w = w / w.sum()
        a = roc_auc_score(y, mats @ w)
        if a > best_auc:
            best_auc, best_w = a, w
    print("10-fold blend", dict(zip(names, best_w)), best_auc)
    pred = np.clip(sum(best_w[i] * te[n] for i, n in enumerate(names)), 1e-7, 1 - 1e-7)
    pd.DataFrame({"id": test["id"], TARGET: pred}).to_csv(ROOT / "submission_own_10fold.csv", index=False)
    print("wrote submission_own_10fold.csv")
else:
    print("skip own 10-fold (set RUN_OWN_10FOLD = True to train)")


## C. 公开 135 列 stacking 与 R3（原 `stack_ours.py` / `push_meta.py`）

- **135 only**：74+50+Paiky5折6列+5 FM，logit Newton LR，OOF 约 0.96966–0.96978
- **R3**：再加 19 extras + 4 个来源分组 LR，OOF **0.969950**

最终提交是在 R3 上再加 mask 与 student，见 `Mask_Student.ipynb`。本节只打印 OOF，不写提交。


In [ ]:
from scipy.special import expit, logit

LIB = ROOT / "oof_lib" / "oof"
WEAK = ROOT / "oof_extra" / "weak50"
PAIKY = ROOT / "oof_extra" / "paiky11"
FM = ROOT / "oof_extra" / "fm"
PAIKY_5F = ("v10_tabm", "v13_lookup", "v14_lookup_bag", "v15_lookup_wide", "v16_lookup_aug", "v17_realmlp")
FM_NAMES = ("fmplr", "fmnum", "fmdeep", "fmwide", "fmpure")
SLICES = {"lib74": (0, 74), "weak50": (74, 124), "paiky": (124, 130), "fm": (130, 135)}
C = 1.0

def clip01(a):
    return np.clip(a, 1e-6, 1.0 - 1e-6)

def to_logit(P):
    return logit(clip01(P))

def _xtwx(X, w, C, chunk=65536):
    n, d = X.shape
    H = np.zeros((d, d), dtype=np.float64)
    for i in range(0, n, chunk):
        sl = slice(i, min(i + chunk, n))
        Xi = X[sl]
        H += (Xi * w[sl, None]).T @ Xi
    return H * C

def fit_logreg(X, y, C=C, max_iter=40):
    n, d = X.shape
    Xb = np.column_stack([X, np.ones(n, dtype=np.float64)])
    w = np.zeros(d + 1, dtype=np.float64)
    y = y.astype(np.float64)
    eye = np.eye(d + 1, dtype=np.float64)
    eye[-1, -1] = 0.0
    for _ in range(max_iter):
        z = np.clip(Xb @ w, -30.0, 30.0)
        p = expit(z)
        g = C * (Xb.T @ (p - y)) + (w * np.r_[np.ones(d), 0.0])
        H = _xtwx(Xb, p * (1.0 - p), C) + eye
        try:
            step = np.linalg.solve(H, g)
        except np.linalg.LinAlgError:
            step = np.linalg.lstsq(H, g, rcond=None)[0]
        w = w - step
        if float(np.max(np.abs(step))) < 1e-7:
            break
    return w

def predict_proba(X, w):
    return expit(np.clip(X @ w[:-1] + w[-1], -30.0, 30.0))

def stack_ready(X, y, Xt, tag):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(y), dtype=np.float64)
    pred = np.zeros(len(Xt), dtype=np.float64)
    for i, (tr, va) in enumerate(skf.split(X, y), 1):
        w = fit_logreg(X[tr], y[tr], C=C)
        oof[va] = predict_proba(X[va], w)
        pred += predict_proba(Xt, w) / 5
        print(f"  [{tag}] fold {i} {roc_auc_score(y[va], oof[va]):.6f}", flush=True)
    auc = float(roc_auc_score(y, oof))
    print(f"  [{tag}] OOF {auc:.6f}", flush=True)
    return oof, pred, auc

def load_public(n_train, n_test, train_ids):
    oof_c, te_c, names = [], [], []
    for op in sorted(LIB.glob("oof_*.npy")):
        name = op.name[len("oof_") : -4]
        tp = LIB / f"test_{name}.npy"
        if not tp.exists():
            continue
        o = np.load(op).astype(np.float64).reshape(-1)
        t = np.load(tp).astype(np.float64).reshape(-1)
        if o.shape[0] != n_train or t.shape[0] != n_test or not (np.isfinite(o).all() and np.isfinite(t).all()):
            continue
        names.append(name); oof_c.append(o); te_c.append(t)
    Xw = np.load(WEAK / "oof.npy").astype(np.float64)
    Tw = np.load(WEAK / "test.npy").astype(np.float64)
    po, pt = pd.read_csv(PAIKY / "oof_predictions.csv"), pd.read_csv(PAIKY / "test_predictions.csv")
    assert (po["id"].values == train_ids).all()
    Xp, Tp = po[list(PAIKY_5F)].to_numpy(np.float64), pt[list(PAIKY_5F)].to_numpy(np.float64)
    Xfm = np.column_stack([np.load(FM / f"oof_{n}.npy").astype(np.float64).reshape(-1) for n in FM_NAMES])
    Tfm = np.column_stack([np.load(FM / f"test_{n}.npy").astype(np.float64).reshape(-1) for n in FM_NAMES])
    X = np.column_stack(oof_c + [Xw, Xp, Xfm])
    Xt = np.column_stack(te_c + [Tw, Tp, Tfm])
    names = names + [f"weak_{i:02d}" for i in range(50)] + list(PAIKY_5F) + list(FM_NAMES)
    return X, Xt, names

def extras(P, names):
    mean, std = P.mean(1), P.std(1)
    p10, p90 = np.quantile(P, 0.10, axis=1), np.quantile(P, 0.90, axis=1)
    row = np.column_stack([to_logit(mean), std, to_logit(p10), to_logit(p90)])
    gcols = []
    for lo, hi in SLICES.values():
        g = P[:, lo:hi]
        gcols += [to_logit(g.mean(1)), g.std(1)]
    rules = {
        "fam_lookup": lambda n: "lookup" in n,
        "fam_tabm": lambda n: "tabm" in n,
        "fam_lat": lambda n: n.startswith("lat") or "lattice" in n or n.startswith("latr") or n.startswith("lattri") or n.startswith("latwide") or n.startswith("latmax"),
        "fam_gbdt": lambda n: any(k in n for k in ("lgbm", "lgb", "xgb", "cat", "hgb", "et", "rf")),
        "fam_nn": lambda n: any(k in n for k in ("mlp", "realmlp", "nn2", "resnet", "tabnet", "rmlp")),
        "fam_fm": lambda n: n.startswith("fm"),
        "fam_naji": lambda n: n.startswith("naji"),
    }
    fcols = []
    for _, pred in rules.items():
        idx = [i for i, n in enumerate(names) if pred(n)]
        fcols.append(to_logit(P[:, idx].mean(1)) if idx else np.zeros(len(P)))
    return np.column_stack([row, np.column_stack(gcols), np.column_stack(fcols)])

def hierarchical(P, Pt, y):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    Ho = np.zeros((len(y), len(SLICES)))
    Ht = np.zeros((len(Pt), len(SLICES)))
    for i, (tr, va) in enumerate(skf.split(P, y), 1):
        fold_te = []
        for j, (lo, hi) in enumerate(SLICES.values()):
            w = fit_logreg(to_logit(P[tr, lo:hi]), y[tr], C=C)
            Ho[va, j] = predict_proba(to_logit(P[va, lo:hi]), w)
            fold_te.append(predict_proba(to_logit(Pt[:, lo:hi]), w))
        Ht += np.column_stack(fold_te) / 5
        print(f"  [hier] fold {i}", flush=True)
    return Ho, Ht

RUN_PUBLIC_STACK = False  # 约 10 分钟；只打印 OOF

if RUN_PUBLIC_STACK:
    train = pd.read_csv(DATA_DIR / "train.csv")
    test = pd.read_csv(DATA_DIR / "test.csv")
    y = train[TARGET].astype(np.int32).values
    P, T, names = load_public(len(train), len(test), train["id"].values)
    assert P.shape[1] == 135
    print("== 135 only ==")
    stack_ready(to_logit(P), y, to_logit(T), "R0")
    Ex, Ext = extras(P, names), extras(T, names)
    print("== R3 extras + hier ==")
    Ho, Ht = hierarchical(P, T, y)
    stack_ready(np.hstack([to_logit(P), Ex, to_logit(Ho)]), y,
                np.hstack([to_logit(T), Ext, to_logit(Ht)]), "R3")
else:
    print("skip public stack (set RUN_PUBLIC_STACK = True)")
